In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable

from torch.utils.data import Dataset,DataLoader

from tokenizers import ByteLevelBPETokenizer
from tokenizers.trainers import BpeTrainer
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers, processors
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace

from torchinfo import summary
import math


САМ МИПЛ

In [10]:
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)


tokenizer.decoder = decoders.ByteLevel()
trainer = BpeTrainer(
    vocab_size=15000,
    special_tokens=["[PAD]", "[UNK]", "[SOS]", "[EOS]"],
    min_frequency=2
)

files = ["VocabText.txt"]
tokenizer.train(files, trainer)

sos_token_id = tokenizer.token_to_id("[SOS]")
eos_token_id = tokenizer.token_to_id("[EOS]")

MAX_LEN = 512
tokenizer.enable_padding(length=MAX_LEN)
tokenizer.enable_truncation(max_length=MAX_LEN)
tokenizer.save("miple_tokenizer.json")



In [11]:
class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads
        self.qkv = nn.Linear(embed_size, embed_size * 3)
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        qkv = qkv.reshape(B, T, 3, self.heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.tril(torch.ones(T, T)).to(x.device)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.fc_out(out)

In [12]:
class Block(nn.Module):
    def __init__(self, embed_size, heads, dropout=0.1):
        super().__init__()
        self.attn = SelfAttention(embed_size, heads)
        self.ln1 = nn.LayerNorm(embed_size)
        self.ff = nn.Sequential(
            nn.Linear(embed_size, 4 * embed_size),
            nn.GELU(),
            nn.Linear(4 * embed_size, embed_size),
            nn.Dropout(dropout),
        )
        self.ln2 = nn.LayerNorm(embed_size)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [13]:
class MipleModel(nn.Module):
    def __init__(self, vocab_size, embed_size, block_size, heads, decisions_count, emotions_count, market_features, n_layers, dropout=0.1):
        super(MipleModel, self).__init__()
        self.block_size = block_size
        self.token_embed = nn.Embedding(vocab_size, embed_size)
        self.pos_embed = nn.Embedding(block_size, embed_size)

        self.blocks = nn.Sequential(*[Block(embed_size, heads, dropout) for _ in range(n_layers)])

        self.market_proj = nn.Linear(market_features, embed_size)
        
        combined_input_size = (embed_size * 3) + emotions_count

        self.buy_decision = nn.Sequential(
            nn.Linear(combined_input_size, embed_size),
            nn.ReLU(),
            nn.Linear(embed_size, embed_size),
            nn.Linear(embed_size, decisions_count),
        )

        self.emotions_choicer = nn.Linear(combined_input_size, emotions_count)

        self.ln_f = nn.LayerNorm(embed_size)
        self.fc = nn.Linear(embed_size, vocab_size)

    def forward(self, market_seq, prompt_seq, news_seq, current_emotions):
        B, T = prompt_seq.shape
        

        token_embeddings = self.token_embed(prompt_seq)
        position_embeddings = self.pos_embed(torch.arange(T, device=prompt_seq.device))
        x = token_embeddings + position_embeddings
        x = self.blocks(x)


        market_emb = self.market_proj(market_seq)
        news_emb = self.token_embed(news_seq) 

        market_emb_mean = torch.mean(market_emb, dim=1)
        news_emb_mean = torch.mean(news_emb, dim=1)
        
        x_mean = torch.mean(x, dim=1)

        combined = torch.cat((x_mean, market_emb_mean, news_emb_mean, current_emotions), dim=1)

        trade_logits = self.buy_decision(combined)
        new_emotions = self.emotions_choicer(combined)


        x_norm = self.ln_f(x)
        text_logits = self.fc(x_norm) 

        return trade_logits, new_emotions, text_logits

In [14]:
# тестовый прогон
vocab_size = tokenizer.get_vocab_size()
embed_size = 512
block_size = 512
heads = 8
decisions_count = 3
emotions_count = 8
market_features = 50
n_layers = 4

model = MipleModel(vocab_size, embed_size, block_size, heads, decisions_count, emotions_count, market_features, n_layers)

dummy_prompt = torch.randint(0, vocab_size, (2, 128), dtype=torch.long)
dummy_market = torch.randn(2, 50, market_features)
dummy_news = torch.randint(0, vocab_size, (2, 20), dtype=torch.long)
dummy_emotions = torch.randn(2, emotions_count)


trade_logits, new_emotions, text_logits = model(dummy_market,dummy_prompt, dummy_news, dummy_emotions)

print("trade logits shape:", trade_logits.shape)
print("new emotions shape:", new_emotions.shape)
print("text logits shape:", text_logits.shape)

model_stats = summary(
    model, 
    input_data=(dummy_market, dummy_prompt, dummy_news, dummy_emotions),
    col_names=["input_size", "output_size", "num_params", "kernel_size"],
    depth=3
)

print(model_stats)

trade logits shape: torch.Size([2, 3])
new emotions shape: torch.Size([2, 8])
text logits shape: torch.Size([2, 128, 3261])
Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Kernel Shape
MipleModel                               [2, 50, 50]               [2, 3]                    --                        --
├─Embedding: 1-1                         [2, 128]                  [2, 128, 512]             1,669,632                 --
├─Embedding: 1-2                         [128]                     [128, 512]                262,144                   --
├─Sequential: 1-3                        [2, 128, 512]             [2, 128, 512]             --                        --
│    └─Block: 2-1                        [2, 128, 512]             [2, 128, 512]             --                        --
│    │    └─LayerNorm: 3-1               [2, 128, 512]             [2, 128, 512]             1,024                     --
│    │    └─

In [15]:
class FineTuningDataset(Dataset):
    def __init__(self, tokenizer, history, max_len=512):
        super().__init__()
        self.tokenizer = tokenizer
        self.history = history

        self.stocks_data = history['stocks_states']
        self.messages_data = history['messages']
        self.events = history['events']
        self.emotions_data = history['emotions']

        self.max_len = max_len
        self.chat_samples = []

        for i in range(len(self.messages_data)):
            text = f"{self.messages_data[i]['input']} [SOS] {self.messages_data[i]['output']} [EOS]"
            self.chat_samples.append(text)

    def __len__(self):
        return len(self.chat_samples)
            
    def __getitem__(self, index):
        encoded_messages = self.tokenizer.encode(self.chat_samples[index]).ids
        encoded_events = self.tokenizer.encode(self.events[index]).ids

        text_out_ids = torch.tensor(encoded_messages, dtype=torch.long)
        events_out = torch.tensor(encoded_events, dtype=torch.long)

        stocks_out = torch.tensor(self.stocks_data[index], dtype=torch.float)
        emotions_out = torch.tensor(self.emotions_data[index], dtype=torch.float)

        return text_out_ids, events_out, stocks_out, emotions_out